# Activation Patching on the IOI Task

**Indirect Object Identification (IOI)** is the classic mechanistic interpretability task (Wang et al., 2023). In sentences like:

> "When Mary and John went to the store, Mary gave a gift to **___**"

the model should predict **John** (the indirect object), not Mary (the subject).

**Activation patching** lets us identify which model components are responsible for this behavior by swapping activations between a "clean" run and a "corrupted" run.

This notebook covers:
1. Setting up the IOI dataset
2. Measuring logit difference (our behavioral metric)
3. Layer-level patching: which layers matter most?
4. Head-level patching: which attention heads implement the IOI circuit?
5. Ablation experiments
6. Interpreting the results

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.datasets import make_ioi_dataset, to_tokens
from irtk.patching import (
    activation_patch, patch_by_layer, patch_by_head,
    zero_ablate, mean_ablate, ablate_heads,
    make_logit_diff_metric,
)
from irtk import vis

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. The IOI Dataset

Each example has:
- A **clean prompt** where the subject (S) is repeated and the indirect object (IO) is the correct answer
- A **corrupted prompt** where the IO name is replaced with a random name
- The **answer token** (IO) and the **wrong token** (S)

In [ ]:
dataset = make_ioi_dataset(n_prompts=20, tokenizer=tokenizer, seed=42)

print(f"Generated {len(dataset.clean_prompts)} prompt pairs")
print()
for i in range(3):
    io = dataset.io_names[i]
    s = dataset.s_names[i]
    print(f"Example {i+1}:")
    print(f"  Clean:     {dataset.clean_prompts[i]}")
    print(f"  Corrupted: {dataset.corrupted_prompts[i]}")
    print(f"  Answer: {io!r} (token {dataset.answer_tokens[i]}), Wrong: {s!r} (token {dataset.wrong_tokens[i]})")
    print()

## 2. The Logit Difference Metric

We measure behavior by the **logit difference**: `logit(IO) - logit(S)` at the final position.

- Positive = model prefers the correct answer (IO)
- Negative = model prefers the wrong answer (S)
- Zero = model is indifferent

In [ ]:
# Pick one example to work with
idx = 0
clean_tokens = to_tokens(dataset.clean_prompts[idx], tokenizer=tokenizer)
corrupted_tokens = to_tokens(dataset.corrupted_prompts[idx], tokenizer=tokenizer)
answer_token = dataset.answer_tokens[idx]
wrong_token = dataset.wrong_tokens[idx]

# Create metric
metric = make_logit_diff_metric(correct_token=answer_token, wrong_token=wrong_token)

# Measure baseline
clean_logits = model(clean_tokens)
corrupted_logits = model(corrupted_tokens)

clean_ld = metric(clean_logits)
corrupted_ld = metric(corrupted_logits)

print(f"Prompt: {dataset.clean_prompts[idx]}")
print(f"Correct answer: {tokenizer.decode([answer_token])!r}")
print(f"Wrong answer: {tokenizer.decode([wrong_token])!r}")
print(f"\nClean logit diff:     {clean_ld:.3f} (positive = correct!)")
print(f"Corrupted logit diff: {corrupted_ld:.3f}")
print(f"Difference:           {clean_ld - corrupted_ld:.3f}")

## 3. Layer-Level Patching

First, let's see which **layers** are most important. We run the model on the clean input, but at each layer, we swap in the residual stream from the corrupted input. If the logit difference drops, that layer is important.

In [ ]:
layer_results = patch_by_layer(model, clean_tokens, corrupted_tokens, metric)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(model.cfg.n_layers), layer_results)
ax.axhline(y=clean_ld, color='green', linestyle='--', alpha=0.7, label=f'Clean ({clean_ld:.2f})')
ax.axhline(y=corrupted_ld, color='red', linestyle='--', alpha=0.7, label=f'Corrupted ({corrupted_ld:.2f})')
ax.set_xlabel("Layer")
ax.set_ylabel("Logit Diff After Patching")
ax.set_title("Activation Patching by Layer (Residual Stream)")
ax.legend()
ax.set_xticks(range(model.cfg.n_layers))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Identify the most impactful layers
# Patching corrupted into clean: if result drops toward corrupted_ld, that layer matters
impact = clean_ld - layer_results  # positive = patching hurt performance
print("\nLayers with biggest impact (logit diff reduction from patching):")
for l in np.argsort(impact)[::-1][:5]:
    print(f"  Layer {l}: impact = {impact[l]:.3f}")

## 4. Head-Level Patching

Now the exciting part: which **individual attention heads** implement the IOI circuit? We patch each head's output (z) independently.

In [ ]:
head_results = patch_by_head(model, clean_tokens, corrupted_tokens, metric)

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(head_results, aspect='auto', cmap='RdBu_r', 
               vmin=corrupted_ld, vmax=clean_ld)

ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Activation Patching by Head\n(Color: logit diff after patching this head)")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
plt.colorbar(im, ax=ax, label="Logit Diff")

# Annotate heads with the biggest impact
head_impact = clean_ld - head_results
for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        if abs(head_impact[layer, head]) > 0.5:
            ax.text(head, layer, f"{head_impact[layer,head]:.1f}",
                   ha='center', va='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Top heads by impact
print("Top 10 heads by patching impact:")
print("(Positive = patching this head reduces correct behavior)")
print("=" * 50)

flat_impact = head_impact.flatten()
sorted_heads = np.argsort(np.abs(flat_impact))[::-1]
for i in range(min(10, len(sorted_heads))):
    idx = sorted_heads[i]
    layer = idx // model.cfg.n_heads
    head = idx % model.cfg.n_heads
    print(f"  L{layer}H{head}: impact = {head_impact[layer, head]:+.3f}")

## 5. Zero Ablation: Which Components Are Necessary?

Patching tells us what happens when we swap activations. **Ablation** tells us what happens when we *remove* a component entirely.

Zero ablation sets a head's output to zero. If the logit diff drops, the head is necessary for the task.

In [ ]:
# Zero-ablate each head independently
ablation_results = ablate_heads(model, clean_tokens, metric, method="zero")

fig, ax = plt.subplots(figsize=(14, 8))
ablation_impact = clean_ld - ablation_results
im = ax.imshow(ablation_impact, aspect='auto', cmap='RdBu_r')

ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Zero Ablation Impact by Head\n(Red=ablating hurts, Blue=ablating helps)")
ax.set_xticks(range(model.cfg.n_heads))
ax.set_yticks(range(model.cfg.n_layers))
plt.colorbar(im, ax=ax, label="Logit Diff Reduction")

for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        if abs(ablation_impact[layer, head]) > 0.5:
            ax.text(head, layer, f"{ablation_impact[layer,head]:.1f}",
                   ha='center', va='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.show()

# Top heads by ablation impact
print("\nTop heads whose ablation reduces logit diff the most (most necessary):")
flat = ablation_impact.flatten()
for i in np.argsort(flat)[::-1][:10]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: impact = {ablation_impact[l, h]:+.3f}")

## 6. Inspecting Key Heads

Let's look at the attention patterns of the most important heads. In the IOI circuit, we expect to see:
- **Name Mover Heads**: Attend from the final position to the IO name
- **S-Inhibition Heads**: Attend to the S token and inhibit it
- **Previous Token Heads / Induction Heads**: Copy name information

In [ ]:
# Get the full cache to inspect attention patterns
logits, cache = model.run_with_cache(clean_tokens)
clean_labels = [tokenizer.decode([int(t)]) for t in clean_tokens]

# Find the top-3 most impactful heads from patching
flat_impact = np.abs(head_impact).flatten()
top_heads = np.argsort(flat_impact)[::-1][:3]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, head_idx in zip(axes, top_heads):
    layer = head_idx // model.cfg.n_heads
    head = head_idx % model.cfg.n_heads
    
    pattern = np.array(cache[("pattern", layer)])[head]  # [q_pos, k_pos]
    
    im = ax.imshow(pattern, cmap='Blues', vmin=0)
    ax.set_title(f"L{layer}H{head} (impact={head_impact[layer,head]:+.2f})")
    
    if len(clean_labels) <= 15:
        ax.set_xticks(range(len(clean_labels)))
        ax.set_xticklabels(clean_labels, rotation=90, fontsize=7)
        ax.set_yticks(range(len(clean_labels)))
        ax.set_yticklabels(clean_labels, fontsize=7)

plt.suptitle("Attention Patterns of Top Impactful Heads", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Averaging Over Multiple Examples

One example may be noisy. Let's average the patching results over several IOI prompts.

In [ ]:
n_examples = min(10, len(dataset.clean_prompts))
all_head_results = []

for i in range(n_examples):
    ct = to_tokens(dataset.clean_prompts[i], tokenizer=tokenizer)
    crt = to_tokens(dataset.corrupted_prompts[i], tokenizer=tokenizer)
    m = make_logit_diff_metric(
        correct_token=dataset.answer_tokens[i],
        wrong_token=dataset.wrong_tokens[i],
    )
    
    # Need same-length sequences for fair comparison
    if ct.shape[0] != crt.shape[0]:
        continue
    
    hr = patch_by_head(model, ct, crt, m)
    clean_ld_i = m(model(ct))
    # Normalize: fraction of clean logit diff preserved
    corrupted_ld_i = m(model(crt))
    if abs(clean_ld_i - corrupted_ld_i) > 0.1:  # skip degenerate cases
        normalized = (hr - corrupted_ld_i) / (clean_ld_i - corrupted_ld_i)
        all_head_results.append(normalized)

if all_head_results:
    avg_results = np.mean(all_head_results, axis=0)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(avg_results, aspect='auto', cmap='RdBu_r', vmin=0, vmax=1)
    ax.set_xlabel("Head")
    ax.set_ylabel("Layer")
    ax.set_title(f"Average Head Patching Results ({len(all_head_results)} examples)\n"
                 "(1.0 = clean behavior, 0.0 = corrupted behavior)")
    ax.set_xticks(range(model.cfg.n_heads))
    ax.set_yticks(range(model.cfg.n_layers))
    plt.colorbar(im, ax=ax, label="Fraction of Clean Logit Diff")
    plt.tight_layout()
    plt.show()
    
    # Most impactful heads (furthest from 1.0 = most changed by patching)
    deviation = np.abs(1.0 - avg_results)
    print("Heads most affected by patching (averaged):")
    flat = deviation.flatten()
    for i in np.argsort(flat)[::-1][:10]:
        l = i // model.cfg.n_heads
        h = i % model.cfg.n_heads
        print(f"  L{l}H{h}: avg normalized = {avg_results[l, h]:.3f} (deviation = {deviation[l, h]:.3f})")
else:
    print("Not enough valid examples for averaging.")

## Key Takeaways

1. **Activation patching** is a causal method: it tells you what actually matters for the computation, not just what correlates.

2. **The IOI circuit** involves a small number of specialized heads:
   - Name mover heads that copy the IO name to the output
   - S-inhibition heads that suppress the repeated subject name
   - Duplicate token and induction heads that route name information

3. **Layer-level patching** gives a coarse picture; **head-level patching** identifies the specific circuit components.

4. **Ablation vs. patching**: Ablation (zero/mean) tests necessity ("does removing this break things?"). Patching tests sufficiency ("does corrupting this change behavior?").

### API Reference

```python
from irtk.patching import (
    activation_patch,         # Patch specific hook points
    patch_by_layer,           # Patch residual stream at each layer
    patch_by_head,            # Patch each attention head independently
    zero_ablate,              # Set activations to zero
    mean_ablate,              # Replace with mean activation
    ablate_heads,             # Ablate each head independently
    make_logit_diff_metric,   # logit(correct) - logit(wrong)
    make_loss_metric,         # Cross-entropy loss at a position
)

from irtk.datasets import make_ioi_dataset  # Generate IOI prompts
```